"""
REVENUE INDEX NOTEBOOK
=======================
Computes daily production-weighted capture rate revenue index RI(t)
from simulated physical variables.

Formulas:
  T_eff(t)   = T_sim(t) + α · GHI_sim(t)
  f_temp(t)  = 1 + γ · (T_eff(t) − T_STC)
  P(t)       = (GHI_sim(t) / G_STC) · f_temp(t) · PR
  RI(t)      = P(t) · CR_sim(t)
  RI_annual  = Σ_{t∈year} RI(t)


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings("ignore")

In [10]:
data_sim      = Path("../../Data/Simulated")
data_modelled = Path("../../Data/Modelled")
data_raw      = Path("../../Data/Raw")
data_results  = Path("../../Data/Results")
models_path   = Path("../../Code/Models")
data_results.mkdir(parents=True, exist_ok=True)

In [11]:
# ── physical constants ────────────────────────────────────────────────────────
G_STC  = 1000.0   # W/m²   — standard test condition irradiance
T_STC  = 25.0     # °C     — STC reference temperature
GAMMA  = -0.0043   # /°C    — temperature coefficient monocrystalline Si
PR     = 0.80     # —      — performance ratio
 
print("── Step 0: Configuration ────────────────────────────────────────")
print(f"  G_STC  = {G_STC} W/m²")
print(f"  T_STC  = {T_STC}°C")
print(f"  γ      = {GAMMA} /°C")
print(f"  PR     = {PR}")
print(f"  data_sim     : {data_sim.resolve()}")
print(f"  data_results : {data_results.resolve()}")

── Step 0: Configuration ────────────────────────────────────────
  G_STC  = 1000.0 W/m²
  T_STC  = 25.0°C
  γ      = -0.0043 /°C
  PR     = 0.8
  data_sim     : C:\Users\LucasMonero\Documents\data projects\Master Thesis\Project\Data\Simulated
  data_results : C:\Users\LucasMonero\Documents\data projects\Master Thesis\Project\Data\Results


In [12]:
# STEP 1 — Calibrate α from ERA5 hourly data
# ─────────────────────────────────────────────────────────────────────────────
# Regression (no intercept):
#   T_target_daily − T_mean_daily = α · GHI_daily_sum
#
# T_target_daily = Σ(GHI_h · T_h) / Σ(GHI_h)   [irradiance-weighted mean T]
# GHI_daily_sum  = Σ(GHI_h)                      [daily GHI sum Wh/m²]

# columns: time, latitude, longitude, t2m, SSRD_Wm2,
#          CLEAR_SKY_GHI, GHI, GHI_index
era5_hourly_path = None
for candidate in [
        Path("../../Data/Cleaned/df_bologna_cleaned.parquet"),
        data_raw      / "df_bologna_cleaned.parquet",
        data_modelled / "df_bologna_cleaned.parquet",]:
    if candidate.exists():
        era5_hourly_path = candidate
        break

if era5_hourly_path is None:
    raise FileNotFoundError(
        "df_bologna_cleaned.parquet not found.\n"
        "Expected in Data/Cleaned/ or Data/Raw/\n"
        "Adjust path in Step 1."
    )

df_era5 = pd.read_parquet(era5_hourly_path)
df_era5["datetime"] = pd.to_datetime(df_era5["time"])
print(f"  ERA5 loaded : {len(df_era5):,} rows")
print(f"  Date range  : {df_era5['datetime'].min()} → "
      f"{df_era5['datetime'].max()}")

# fixed column names from df_bologna_cleaned
ghi_col = "GHI"    # W/m² — observed GHI
t_col   = "t2m"    # °C   — 2m ambient temperature

# confirm temperature is in °C
if df_era5[t_col].mean() > 100:
    df_era5[t_col] = df_era5[t_col] - 273.15
    print(f"  t2m converted from K to °C")
else:
    print(f"  t2m in °C  (mean={df_era5[t_col].mean():.2f}°C)")

print(f"  GHI col     : '{ghi_col}'  "
      f"mean={df_era5[ghi_col].mean():.1f} W/m²")


  ERA5 loaded : 184,079 rows
  Date range  : 2005-01-01 01:00:00 → 2025-12-31 23:00:00
  t2m in °C  (mean=13.30°C)
  GHI col     : 'GHI'  mean=170.2 W/m²


In [13]:
df_era5.head(10)

,index,time,latitude,longitude,t2m,SSRD_Wm2,CLEAR_SKY_GHI,GHI,GHI_index,datetime
0,0,2005-01-01 01:00:00,45.5,11.25,0.368042,0.000000,0.000000,0.000000,NaN,2005-01-01 01:00:00
1,1,2005-01-01 02:00:00,45.5,11.25,0.558685,0.000000,0.000000,0.000000,NaN,2005-01-01 02:00:00
2,2,2005-01-01 03:00:00,45.5,11.25,0.467987,0.000000,0.000000,0.000000,NaN,2005-01-01 03:00:00
3,3,2005-01-01 04:00:00,45.5,11.25,0.576385,0.000000,0.000000,0.000000,NaN,2005-01-01 04:00:00
4,4,2005-01-01 05:00:00,45.5,11.25,0.582214,0.000000,0.000000,0.000000,NaN,2005-01-01 05:00:00
5,5,2005-01-01 06:00:00,45.5,11.25,0.628815,0.000000,0.000000,0.000000,NaN,2005-01-01 06:00:00
6,6,2005-01-01 07:00:00,45.5,11.25,-0.008026,0.000000,0.054937,0.054937,1.000000,2005-01-01 07:00:00
7,7,2005-01-01 08:00:00,45.5,11.25,0.815399,41.955555,59.074631,59.074631,1.000000,2005-01-01 08:00:00
8,8,2005-01-01 09:00:00,45.5,11.25,2.666901,157.315552,194.115555,194.023499,0.999526,2005-01-01 09:00:00
9,9,2005-01-01 10:00:00,45.5,11.25,7.044800,261.191101,307.682556,299.007507,0.971805,2005-01-01 10:00:00


In [15]:
# ── daily aggregation ─────────────────────────────────────────────────────
df_era5["date_only"] = df_era5["datetime"].dt.date
df_day = df_era5[df_era5[ghi_col] > 0].copy()
 
daily = (
        df_day.groupby("date_only")
        .apply(lambda g: pd.Series({
            "GHI_daily_sum": g[ghi_col].sum(),
            "T_target":      np.average(g[t_col], weights=g[ghi_col]),
        }))
        .reset_index()
    )
daily_mean = (
        df_era5.groupby("date_only")[t_col]
        .mean().reset_index()
        .rename(columns={t_col: "T_mean_daily", "date_only": "date_only"})
    )
daily = daily.merge(daily_mean, on="date_only", how="inner")
daily["T_diff"] = daily["T_target"] - daily["T_mean_daily"]
 
MIN_GHI   = 500
daily_fit = daily[daily["GHI_daily_sum"] > MIN_GHI].copy()
print(f"  Days for regression (GHI>{MIN_GHI}): {len(daily_fit)}")
 
# no-intercept OLS: T_diff = α · GHI_daily_sum
x     = daily_fit["GHI_daily_sum"].values
y     = daily_fit["T_diff"].values
alpha = float(np.dot(x, y) / np.dot(x, x))
y_hat = alpha * x
ss_res   = float(np.sum((y - y_hat)**2))
ss_tot   = float(np.sum((y - y.mean())**2))
alpha_r2 = float(1 - ss_res / ss_tot)
 
print(f"  α  = {alpha:.6f} °C per Wh/m²")
print(f"  R² = {alpha_r2:.4f}")
print(f"  Example: GHI=6000 Wh/m² → uplift = {alpha*6000:.2f}°C")

  Days for regression (GHI>500): 7504
  α  = 0.000494 °C per Wh/m²
  R² = -0.7519
  Example: GHI=6000 Wh/m² → uplift = 2.96°C
